In [1]:
import pandas as pd
import numpy as np

cols = ["loan_amnt", "funded_amnt", "term", "int_rate", "installment","grade", "sub_grade", "emp_length", "home_ownership", "annual_inc",
    "verification_status", "issue_d", "loan_status", "purpose","addr_state", "dti", "delinq_2yrs", "earliest_cr_line",
        "fico_range_low", "fico_range_high", "open_acc", "revol_util",
    "total_acc", "total_rec_prncp"]
df = pd.read_csv(r"C:\Users\prana\OneDrive\Desktop\loan-default-risk\data\raw\accepted_2007_to_2018Q4.csv",usecols=cols, low_memory=False)
df.head()
df.info

<bound method DataFrame.info of          loan_amnt  funded_amnt        term  int_rate  installment grade  \
0           3600.0       3600.0   36 months     13.99       123.03     C   
1          24700.0      24700.0   36 months     11.99       820.28     C   
2          20000.0      20000.0   60 months     10.78       432.66     B   
3          35000.0      35000.0   60 months     14.85       829.90     C   
4          10400.0      10400.0   60 months     22.45       289.91     F   
...            ...          ...         ...       ...          ...   ...   
2260696    40000.0      40000.0   60 months     10.49       859.56     B   
2260697    24000.0      24000.0   60 months     14.49       564.56     C   
2260698    14000.0      14000.0   60 months     14.49       329.33     C   
2260699        NaN          NaN         NaN       NaN          NaN   NaN   
2260700        NaN          NaN         NaN       NaN          NaN   NaN   

        sub_grade emp_length home_ownership  annual_inc

In [4]:
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
df["default_flag"] = (df["loan_status"] == "Charged Off").astype(int)
df["term_months"] = df["term"].str.extract(r"(\d+)").astype(int)
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["earliest_cr_line"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y")
df["issue_year"] = df["issue_d"].dt.year
df["credit_history_years"] = ((df["issue_d"] - df["earliest_cr_line"]).dt.days / 365.25).round(1)
df["fico_avg"] = (df["fico_range_low"] + df["fico_range_high"]) / 2

emp_map = {"< 1 year": "0-1 yr", "1 year": "1-3 yrs", "2 years": "1-3 yrs",
           "3 years": "1-3 yrs", "4 years": "4-6 yrs", "5 years": "4-6 yrs",
           "6 years": "4-6 yrs", "7 years": "7-9 yrs", "8 years": "7-9 yrs",
           "9 years": "7-9 yrs", "10+ years": "10+ yrs"}
df["emp_group"] = df["emp_length"].map(emp_map).fillna("Unknown")
df["home_ownership"] = df["home_ownership"].replace({"ANY": "OTHER", "NONE": "OTHER"})
df = df.dropna(subset=["dti", "revol_util", "annual_inc"])
df = df[(df["annual_inc"] > 0) & (df["annual_inc"] < 500_000) & (df["dti"] >= 0) & (df["dti"] <= 60)]
df["revol_util"] = df["revol_util"].clip(upper=100)
df["loss_amount"] = np.where(df["default_flag"] == 1,df["funded_amnt"] - df["total_rec_prncp"], 0)
print(df.shape, df["default_flag"].mean().round(3))

(1340221, 31) 0.2


In [5]:
def band(series, bins, labels):
    return pd.cut(series, bins=bins, labels=labels, right=False).astype(str)
df["fico_band"] = band(df["fico_avg"], [0, 660, 700, 740, 780, 900],["1) <660", "2) 660-699", "3) 700-739", "4) 740-779", "5) 780+"])
df["income_band"] = band(df["annual_inc"], [0, 40000, 60000, 80000, 100000, 150000, 1e9],["1) <40K", "2) 40-60K", "3) 60-80K", "4)
80-100K", "5) 100-150K", "6) 150K+"])
df["dti_band"] = band(df["dti"], [0, 10, 20, 30, 40, 100],["1) 0-10", "2) 10-20", "3) 20-30", "4) 30-40", "5) 40+"])
df["loan_band"] = band(df["loan_amnt"], [0, 5000, 10000, 20000, 30000, 1e9],["1) <5K", "2) 5-10K", "3) 10-20K", "4) 20-30K", "5) 30K+"])
df["country"] = "United States"

In [6]:
print(df.groupby("grade")["default_flag"].mean().round(3))
print(df.groupby("term_months")["default_flag"].mean().round(3))
print(df.groupby("fico_band")["default_flag"].mean().round(3))
print(df.groupby("issue_year")["default_flag"].agg(["count", "mean"]).round(3))

grade
A    0.060
B    0.134
C    0.224
D    0.304
E    0.385
F    0.452
G    0.499
Name: default_flag, dtype: float64
term_months
36    0.160
60    0.325
Name: default_flag, dtype: float64
fico_band
1) <660       0.000
2) 660-699    0.236
3) 700-739    0.161
4) 740-779    0.106
5) 780+       0.071
Name: default_flag, dtype: float64
             count   mean
issue_year               
2007           250  0.176
2008          1550  0.157
2009          4689  0.126
2010         11481  0.128
2011         21694  0.152
2012         53264  0.162
2013        134599  0.156
2014        222717  0.185
2015        374758  0.202
2016        292104  0.233
2017        167708  0.231
2018         55407  0.157


In [8]:
final_cols = ["loan_amnt", "funded_amnt", "term_months", "int_rate", "grade", "sub_grade","emp_group", "home_ownership", "annual_inc",
"verification_status", "issue_d","issue_year", "purpose", "addr_state", "country", "dti", "fico_avg","credit_history_years",
"revol_util", "loan_status", "default_flag","loss_amount", "fico_band", "income_band", "dti_band", "loan_band"]
df[final_cols].to_csv(r"C:\Users\prana\OneDrive\Desktop\loan-default-risk\data\clean/lending_club_clean.csv", index=False)